In [2]:
import pandas as pd
import polars as pl
import numpy as np
import os
from pathlib import Path
import pandas as pd
import re, pathlib
from datetime import datetime

In [3]:
SCRIPT_DIR   = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
PROJECT_ROOT = SCRIPT_DIR.parent          # edit if your notebook is elsewhere

DATA_DIR       = PROJECT_ROOT / "data/"
SIMULATION_DIR = DATA_DIR / "simulations/"          # folder with ATTRIBUTE_* and wide SSP file
TORNADO_SIM_DIR = SIMULATION_DIR /"data_for_LSU"

In [4]:
wide_inputs_outputs_df = pd.read_csv(os.path.join(TORNADO_SIM_DIR, "louisiana_leap.csv"))
wide_inputs_outputs_df

,primary_id,region,time_period,area_agrc_crops_bevs_and_spices,area_agrc_crops_cereals,area_agrc_crops_fibers,area_agrc_crops_fruits,area_agrc_crops_herbs_and_other_perennial_crops,area_agrc_crops_nuts,area_agrc_crops_other_annual,...,yf_agrc_nuts_tonne_ha,yf_agrc_other_annual_tonne_ha,yf_agrc_other_woody_perennial_tonne_ha,yf_agrc_pulses_tonne_ha,yf_agrc_rice_tonne_ha,yf_agrc_sugar_cane_tonne_ha,yf_agrc_tubers_tonne_ha,yf_agrc_vegetables_and_vines_tonne_ha,yf_lndu_supremum_pastures_tonne_per_ha,emission_co2e_co2_agrc_soil_carbon_organic_soils
0,0,louisiana,6,0,358172.544886,66420.427470,78.095144,77086.928193,6535.540948,1.123806e+06,...,2.949341,6.424512,0,3.494935,8.512478,85.248332,39.935028,30.906144,92.81,0
1,0,louisiana,7,0,356696.043492,66146.621297,77.773211,76769.151304,6508.599365,1.119173e+06,...,2.949341,6.177415,0,3.474771,8.253027,87.719298,39.935028,30.906144,92.81,0
2,0,louisiana,8,0,355221.860914,65873.245131,77.451784,76451.873477,6481.700093,1.114548e+06,...,2.949341,5.189029,0,2.688411,8.475414,83.271559,39.935028,30.906144,92.81,0
3,0,louisiana,9,0,353750.075712,65600.313541,77.130879,76135.111621,6454.844566,1.109930e+06,...,2.949341,6.319971,0,3.416092,8.422668,89.334930,39.935028,30.906144,92.81,0
4,0,louisiana,10,0,352280.764654,65327.840762,76.810514,75818.882257,6428.034184,1.105320e+06,...,2.949341,6.319971,0,3.416092,8.422668,89.334930,39.935028,30.906144,92.81,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
685,91091,louisiana,31,0,322115.206746,59733.863000,70.233283,69326.563877,5877.606069,1.010672e+06,...,2.949341,6.319971,0,3.416092,8.422668,89.334930,39.935028,30.906144,92.81,0
686,91091,louisiana,32,0,320716.481165,59474.479771,69.928308,69025.526123,5852.083654,1.006283e+06,...,2.949341,6.319971,0,3.416092,8.422668,89.334930,39.935028,30.906144,92.81,0
687,91091,louisiana,33,0,319321.564611,59215.802899,69.624163,68725.308159,5826.630742,1.001907e+06,...,2.949341,6.319971,0,3.416092,8.422668,89.334930,39.935028,30.906144,92.81,0
688,91091,louisiana,34,0,317930.498879,58957.840132,69.320858,68425.918980,5801.248096,9.975420e+05,...,2.949341,6.319971,0,3.416092,8.422668,89.334930,39.935028,30.906144,92.81,0


In [5]:
# Get the subsector total variables
industry_value_fuel_vars = [c for c in wide_inputs_outputs_df.columns if "energy_demand_inen_" in c]

In [6]:
# Filter to only production columns avoiding "subsector" total columns
industrial_production_df = wide_inputs_outputs_df[["primary_id", "time_period"] + industry_value_fuel_vars]
industrial_production_df

,primary_id,time_period,energy_demand_inen_agriculture_and_livestock,energy_demand_inen_cement,energy_demand_inen_chemicals,energy_demand_inen_electronics,energy_demand_inen_glass,energy_demand_inen_lime_and_carbonite,energy_demand_inen_metals,energy_demand_inen_mining,...,energy_demand_inen_recycled_glass,energy_demand_inen_recycled_metals,energy_demand_inen_recycled_paper,energy_demand_inen_recycled_plastic,energy_demand_inen_recycled_rubber_and_leather,energy_demand_inen_recycled_textiles,energy_demand_inen_recycled_wood,energy_demand_inen_rubber_and_leather,energy_demand_inen_textiles,energy_demand_inen_wood
0,0,6,68.057201,0,139.207830,0.176244,0.050161,0,0.000000,2.450789,...,0,0,0,0,0,0,0,9.714276,0,53.156776
1,0,7,68.185035,0,137.738402,0.174383,0.049643,0,0.000000,2.424919,...,0,0,0,0,0,0,0,9.860646,0,53.327575
2,0,8,63.992491,0,136.333375,0.172604,0.049147,0,0.000000,2.400183,...,0,0,0,0,0,0,0,10.022317,0,53.523605
3,0,9,68.721143,0,135.007960,0.170926,0.048681,0,0.000000,2.376849,...,0,0,0,0,0,0,0,10.198484,0,53.742789
4,0,10,68.398070,0,133.782097,0.169374,0.048250,0,0.000000,2.355267,...,0,0,0,0,0,0,0,10.385697,0,53.978378
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
685,91091,31,61.988912,0,123.160817,0.155927,0.044550,0,3.646836,2.168277,...,0,0,0,0,0,0,0,14.128739,0,54.420409
686,91091,32,61.700979,0,124.360948,0.157447,0.044986,0,3.614033,2.189406,...,0,0,0,0,0,0,0,14.383657,0,54.622253
687,91091,33,61.414558,0,125.744347,0.159198,0.045488,0,3.582547,2.213761,...,0,0,0,0,0,0,0,14.647050,0,54.845141
688,91091,34,61.129631,0,127.315752,0.161188,0.046057,0,3.552364,2.241426,...,0,0,0,0,0,0,0,14.918939,0,55.089081


In [7]:
# 2) Define your fuels and sectors
relevant_fuels = ['biomass', 
                    'coal', 
                    'coke', 
                    'diesel', 
                    'electricity',
                    'furnace_gas',
                    'gasoline', 
                    'hydrocarbon_gas_liquids',
                    'hydrogen',
                    'kerosene',
                    'natural_gas',
                    'oil']

sectors = ['agriculture_and_livestock',
           'cement',
           'chemicals',
           'electronics',
           'glass',
           'lime_and_carbonite',
           'metals',
           'mining',
           'other_product_manufacturing',
           'paper',
           'plastic',
           'recycled_glass',
           'recycled_metals',
           'recycled_paper',
           'recycled_plastic',
           'recycled_rubber_and_leather',
           'recycled_textiles',
           'recycled_wood',
           'rubber_and_leather',
           'textiles',
           'wood']

In [8]:
# 4) Industrial cost parameters
capex_industrial_electricity = 92666.6 * 21
capex_industrial_other       = 92666.6 * 12
opex_industrial_electricity  = 92666.6 * 2.5
opex_industrial_other        = 92666.6 * 4.5
capex_multiplier_efficiency = 10000000
opex_multiplier_efficiency = 0

In [9]:
# 5) Loop over fuels and sectors
# Use the original dataframe directly
# Initialize results dataframe
ind_fuel_demand_by_sector = pd.DataFrame({
    'primary_id': wide_inputs_outputs_df['primary_id'],
    'time_period': wide_inputs_outputs_df['time_period']
}, index=wide_inputs_outputs_df.index)

# Loop over fuels and sectors
for fuel in relevant_fuels:
    # efficiency column for this fuel
    eff_cols = [c for c in wide_inputs_outputs_df.columns
                if c.startswith(f'efficfactor_enfu_industrial_energy_fuel_{fuel}')]
    if not eff_cols:
        continue
    fuel_efficiency = wide_inputs_outputs_df[eff_cols[0]]

    for sector in sectors:
        sector_dem_cols = [c for c in wide_inputs_outputs_df.columns
                           if f'energy_demand_inen_{sector}' in c]
        sector_fuel_fraction_cols = [c for c in wide_inputs_outputs_df.columns
                                     if f'frac_inen_energy_{sector}_{fuel}' in c]

        if sector_dem_cols and sector_fuel_fraction_cols:
            sector_total_demand = wide_inputs_outputs_df[sector_dem_cols[0]]
            sector_fuel_fraction = wide_inputs_outputs_df[sector_fuel_fraction_cols[0]]

            if (sector_fuel_fraction * sector_total_demand).sum() > 0 or fuel == 'electricity':
                sector_fuel_demand = sector_fuel_fraction * sector_total_demand
                ind_fuel_demand_by_sector[f'energy_demand_{sector}_{fuel}'] = sector_fuel_demand

                # CAPEX/OPEX
                if fuel == 'electricity':
                    ind_fuel_demand_by_sector[f'energy_demand_capex_{sector}_{fuel}'] = (
                        sector_fuel_demand * capex_industrial_electricity
                    )
                    ind_fuel_demand_by_sector[f'energy_demand_opex_{sector}_{fuel}'] = (
                        sector_fuel_demand * opex_industrial_electricity
                    )
                else:
                    ind_fuel_demand_by_sector[f'energy_demand_capex_{sector}_{fuel}'] = (
                        sector_fuel_demand * capex_industrial_other
                    )
                    ind_fuel_demand_by_sector[f'energy_demand_opex_{sector}_{fuel}'] = (
                        sector_fuel_demand * opex_industrial_other
                    )

                # Fuel consumed
                sector_fuel_consumed = sector_fuel_demand / fuel_efficiency

                # Baseline = first time_period per primary_id
                sector_fuel_consumed_baseline = (
                    sector_fuel_consumed.groupby(wide_inputs_outputs_df['primary_id'])
                                        .transform('first')
                )

                # Change relative to baseline
                sector_change_in_fuel_consumed = (
                    sector_fuel_consumed_baseline - sector_fuel_consumed
                )

                # Save results
                ind_fuel_demand_by_sector[f'efficiency_energy_saving_{sector}_{fuel}'] = (
                    sector_change_in_fuel_consumed
                )
                ind_fuel_demand_by_sector[f'efficiency_capex_{sector}_{fuel}'] = (
                    sector_change_in_fuel_consumed * capex_multiplier_efficiency
                )
                ind_fuel_demand_by_sector[f'efficiency_opex_{sector}_{fuel}'] = (
                    sector_change_in_fuel_consumed * opex_multiplier_efficiency
                )


C:\Users\pkane\AppData\Local\Temp\ipykernel_21572\3509172589.py:63: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  ind_fuel_demand_by_sector[f'efficiency_energy_saving_{sector}_{fuel}'] = (
C:\Users\pkane\AppData\Local\Temp\ipykernel_21572\3509172589.py:66: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  ind_fuel_demand_by_sector[f'efficiency_capex_{sector}_{fuel}'] = (
C:\Users\pkane\AppData\Local\Temp\ipykernel_21572\3509172589.py:69: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `fram

In [10]:
ind_fuel_demand_by_sector.head()

,primary_id,time_period,energy_demand_chemicals_coal,energy_demand_capex_chemicals_coal,energy_demand_opex_chemicals_coal,efficiency_energy_saving_chemicals_coal,efficiency_capex_chemicals_coal,efficiency_opex_chemicals_coal,energy_demand_glass_coal,energy_demand_capex_glass_coal,...,energy_demand_opex_rubber_and_leather_oil,efficiency_energy_saving_rubber_and_leather_oil,efficiency_capex_rubber_and_leather_oil,efficiency_opex_rubber_and_leather_oil,energy_demand_wood_oil,energy_demand_capex_wood_oil,energy_demand_opex_wood_oil,efficiency_energy_saving_wood_oil,efficiency_capex_wood_oil,efficiency_opex_wood_oil
0,0,6,1.228421,1.366003e+06,512251.312034,0.000000,0.000000e+00,0.0,0.007378,8204.324615,...,47857.783603,0.000000,0.000000,0.0,1.737080,1.931631e+06,724361.759506,0.000000,0.000000e+00,0.0
1,0,7,0.578123,6.428723e+05,241077.120813,1.087692,1.087692e+07,0.0,0.005446,6055.923881,...,46654.283411,0.004446,44460.209782,0.0,1.729273,1.922950e+06,721106.360580,0.019650,1.965027e+05,0.0
2,0,8,0.055053,6.121924e+04,22957.214954,1.956347,1.956347e+07,0.0,0.004431,4926.852853,...,47777.086821,0.001480,14801.400795,0.0,1.677434,1.865305e+06,699489.483409,0.097420,9.742035e+05,0.0
3,0,9,0.000000,0.000000e+00,0.000000,2.047369,2.047369e+07,0.0,0.004268,4745.464314,...,49083.374049,-0.002039,-20392.325894,-0.0,1.612988,1.793641e+06,672615.422208,0.191212,1.912122e+06,0.0
4,0,10,0.000000,0.000000e+00,0.000000,2.047369,2.047369e+07,0.0,0.004183,4651.366306,...,49537.229981,-0.002846,-28457.928747,-0.0,1.548267,1.721671e+06,645626.741376,0.284648,2.846484e+06,0.0


In [11]:
# Sum all the subsector emission columns across axis=1
la_production_total_df = industrial_production_df.copy()
la_production_total_df["production_total"] = la_production_total_df[industry_value_fuel_vars].sum(axis=1)
la_production_total_df.head()

,primary_id,time_period,energy_demand_inen_agriculture_and_livestock,energy_demand_inen_cement,energy_demand_inen_chemicals,energy_demand_inen_electronics,energy_demand_inen_glass,energy_demand_inen_lime_and_carbonite,energy_demand_inen_metals,energy_demand_inen_mining,...,energy_demand_inen_recycled_metals,energy_demand_inen_recycled_paper,energy_demand_inen_recycled_plastic,energy_demand_inen_recycled_rubber_and_leather,energy_demand_inen_recycled_textiles,energy_demand_inen_recycled_wood,energy_demand_inen_rubber_and_leather,energy_demand_inen_textiles,energy_demand_inen_wood,production_total
0,0,6,68.057201,0,139.207830,0.176244,0.050161,0,0.0,2.450789,...,0,0,0,0,0,0,9.714276,0,53.156776,452.932833
1,0,7,68.185035,0,137.738402,0.174383,0.049643,0,0.0,2.424919,...,0,0,0,0,0,0,9.860646,0,53.327575,450.805569
2,0,8,63.992491,0,136.333375,0.172604,0.049147,0,0.0,2.400183,...,0,0,0,0,0,0,10.022317,0,53.523605,444.558321
3,0,9,68.721143,0,135.007960,0.170926,0.048681,0,0.0,2.376849,...,0,0,0,0,0,0,10.198484,0,53.742789,447.457678
4,0,10,68.398070,0,133.782097,0.169374,0.048250,0,0.0,2.355267,...,0,0,0,0,0,0,10.385697,0,53.978378,445.551594


In [12]:
# Keep only the primary_id, time_period, and emission_total columns
la_production_total_df = la_production_total_df[["primary_id", "time_period", "production_total"]]
la_production_total_df.head()

,primary_id,time_period,production_total
0,0,6,452.932833
1,0,7,450.805569
2,0,8,444.558321
3,0,9,447.457678
4,0,10,445.551594


In [13]:
# aggregate data by primary_id summing the emissions
la_production_df_sum_agg = la_production_total_df.groupby(["primary_id"]).mean().reset_index()

# Drop time period column
la_production_df_sum_agg = la_production_df_sum_agg.drop(columns=["time_period"])
la_production_df_sum_agg.head()

,primary_id,production_total
0,0,454.482684
1,70070,398.137913
2,71071,436.980824
3,72072,398.137913
4,73073,436.980824


In [14]:
la_production_df_sum_agg.shape

(23, 2)

In [15]:
industry_cost_vars = [
    c for c in ind_fuel_demand_by_sector.columns
    if c.startswith("energy_demand_capex_") or c.startswith("energy_demand_opex_")
]

In [16]:
ind_fuel_demand_by_sector[industry_cost_vars]

,energy_demand_capex_chemicals_coal,energy_demand_opex_chemicals_coal,energy_demand_capex_glass_coal,energy_demand_opex_glass_coal,energy_demand_capex_mining_coal,energy_demand_opex_mining_coal,energy_demand_capex_other_product_manufacturing_coal,energy_demand_opex_other_product_manufacturing_coal,energy_demand_capex_paper_coal,energy_demand_opex_paper_coal,...,energy_demand_capex_other_product_manufacturing_oil,energy_demand_opex_other_product_manufacturing_oil,energy_demand_capex_paper_oil,energy_demand_opex_paper_oil,energy_demand_capex_plastic_oil,energy_demand_opex_plastic_oil,energy_demand_capex_rubber_and_leather_oil,energy_demand_opex_rubber_and_leather_oil,energy_demand_capex_wood_oil,energy_demand_opex_wood_oil
0,1.366003e+06,512251.312034,8204.324615,3076.621731,2255.227598,845.710349,6859.770409,2572.413904,999159.213788,374684.705171,...,48824.473961,18309.177736,56817.204566,21306.451712,812548.751895,304705.781961,127620.756274,47857.783603,1.931631e+06,724361.759506
1,6.428723e+05,241077.120813,6055.923881,2270.971455,1760.747745,660.280405,6518.579847,2444.467442,0.000000,0.000000,...,51535.889642,19325.958616,0.000000,0.000000,776594.192761,291222.822285,124411.422428,46654.283411,1.922950e+06,721106.360580
2,6.121924e+04,22957.214954,4926.852853,1847.569820,1375.924719,515.971770,6117.427093,2294.035160,0.000000,0.000000,...,54255.324938,20345.746852,0.000000,0.000000,780823.261971,292808.723239,127405.564855,47777.086821,1.865305e+06,699489.483409
3,0.000000e+00,0.000000,4745.464314,1779.549118,1037.181020,388.942883,5744.697100,2154.261413,0.000000,0.000000,...,57089.818492,21408.681934,0.000000,0.000000,787772.727569,295414.772838,130888.997463,49083.374049,1.793641e+06,672615.422208
4,0.000000e+00,0.000000,4651.366306,1744.262365,669.304813,250.989305,5341.825836,2003.184688,0.000000,0.000000,...,60237.605350,22589.102006,0.000000,0.000000,779275.627977,292228.360491,132099.279950,49537.229981,1.721671e+06,645626.741376
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
685,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,49683.356138,18631.258552,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2.876453e+05,107866.994155
686,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,46444.162091,17416.560784,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2.628254e+05,98559.515960
687,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,42964.735817,16111.775931,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2.412636e+05,90473.843340
688,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,39238.852239,14714.569590,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2.228617e+05,83573.141719


In [17]:
# Sum all the subsector emission columns across axis=1
la_production_cost_total_df = ind_fuel_demand_by_sector.copy()
la_production_cost_total_df["production_cost_total"] = la_production_cost_total_df[industry_cost_vars].sum(axis=1)
la_production_cost_total_df.head()

,primary_id,time_period,energy_demand_chemicals_coal,energy_demand_capex_chemicals_coal,energy_demand_opex_chemicals_coal,efficiency_energy_saving_chemicals_coal,efficiency_capex_chemicals_coal,efficiency_opex_chemicals_coal,energy_demand_glass_coal,energy_demand_capex_glass_coal,...,efficiency_energy_saving_rubber_and_leather_oil,efficiency_capex_rubber_and_leather_oil,efficiency_opex_rubber_and_leather_oil,energy_demand_wood_oil,energy_demand_capex_wood_oil,energy_demand_opex_wood_oil,efficiency_energy_saving_wood_oil,efficiency_capex_wood_oil,efficiency_opex_wood_oil,production_cost_total
0,0,6,1.228421,1.366003e+06,512251.312034,0.000000,0.000000e+00,0.0,0.007378,8204.324615,...,0.000000,0.000000,0.0,1.737080,1.931631e+06,724361.759506,0.000000,0.000000e+00,0.0,5.930060e+08
1,0,7,0.578123,6.428723e+05,241077.120813,1.087692,1.087692e+07,0.0,0.005446,6055.923881,...,0.004446,44460.209782,0.0,1.729273,1.922950e+06,721106.360580,0.019650,1.965027e+05,0.0,5.917141e+08
2,0,8,0.055053,6.121924e+04,22957.214954,1.956347,1.956347e+07,0.0,0.004431,4926.852853,...,0.001480,14801.400795,0.0,1.677434,1.865305e+06,699489.483409,0.097420,9.742035e+05,0.0,5.824567e+08
3,0,9,0.000000,0.000000e+00,0.000000,2.047369,2.047369e+07,0.0,0.004268,4745.464314,...,-0.002039,-20392.325894,-0.0,1.612988,1.793641e+06,672615.422208,0.191212,1.912122e+06,0.0,5.877396e+08
4,0,10,0.000000,0.000000e+00,0.000000,2.047369,2.047369e+07,0.0,0.004183,4651.366306,...,-0.002846,-28457.928747,-0.0,1.548267,1.721671e+06,645626.741376,0.284648,2.846484e+06,0.0,5.859564e+08


In [18]:
# Keep only the primary_id, time_period, and emission_total columns
la_production_cost_total_df = la_production_cost_total_df[["primary_id", "time_period", "production_cost_total"]]
la_production_cost_total_df.head()

,primary_id,time_period,production_cost_total
0,0,6,5.930060e+08
1,0,7,5.917141e+08
2,0,8,5.824567e+08
3,0,9,5.877396e+08
4,0,10,5.859564e+08


In [19]:
# aggregate data by primary_id summing the emissions
la_production_cost_df_sum_agg = la_production_cost_total_df.groupby(["primary_id"]).mean().reset_index()

# Drop time period column
la_production_cost_df_sum_agg = la_production_cost_df_sum_agg.drop(columns=["time_period"])
la_production_cost_df_sum_agg.head()

,primary_id,production_cost_total
0,0,6.293694e+08
1,70070,6.458774e+08
2,71071,6.692580e+08
3,72072,6.458774e+08
4,73073,6.692580e+08


In [20]:
la_production_df_sum_agg.shape

(23, 2)

In [21]:
lsu_data = pd.read_csv(os.path.join(TORNADO_SIM_DIR, "lsu_output_leap_cases.csv"))
lsu_data.head()

,time,primary_id,la_value_direct,la_value_indirect,la_value_induced,la_earnings_direct,la_earnings_indirect,la_earnings_induced,la_employment_direct,la_employment_indirect,la_employment_total
0,6,70070,-2.384186e-07,-1.192093e-07,-1.639128e-07,-2.086163e-07,-5.960464e-08,-8.940697e-08,-3.183231e-12,-1.023182e-12,-6.366463e-12
1,7,70070,-3.576279e-07,-1.788139e-07,-2.384186e-07,-2.831221e-07,-8.195639e-08,-1.192093e-07,-4.547474e-12,-1.364242e-12,-8.185452e-12
2,8,70070,-3.576279e-07,4.470348e-08,-1.206994e-06,-3.129244e-07,2.980232e-08,-6.034970e-07,-5.002221e-12,3.410605e-13,-1.637090e-11
3,9,70070,-5.662441e-07,-2.831221e-07,-3.874302e-07,-4.768372e-07,-1.490116e-07,-2.011657e-07,-7.730705e-12,-2.387424e-12,-1.455192e-11
4,10,70070,-8.642673e-07,-4.321337e-07,-5.662441e-07,-7.152557e-07,-2.160668e-07,-3.054738e-07,-1.182343e-11,-3.410605e-12,-2.182787e-11


In [22]:
lsu_data["num_jobs"] = lsu_data[["la_employment_direct", "la_employment_indirect"]].sum(axis=1, min_count=1)


In [23]:
lsu_data["earning_per_job"] = (
    lsu_data[["la_earnings_direct", "la_earnings_indirect"]].sum(axis=1, min_count=1)
    / lsu_data["num_jobs"].replace(0, pd.NA)
)


In [24]:
lsu_data = lsu_data[["primary_id",
               "time",
               "num_jobs",
               "earning_per_job"]]

In [25]:
# aggregate data by primary_id and region by summing the technical cost
lsu_data_agg = lsu_data.groupby(["primary_id"]).mean().reset_index()
lsu_data_agg


,primary_id,time,num_jobs,earning_per_job
0,70070,20.5,24691.438413,79238.771877
1,71071,20.5,16480.194664,76819.453052
2,72072,20.5,11844.307150,95135.541477
3,73073,20.5,7673.895887,66698.88536
4,74074,20.5,2746.090115,69910.918759
5,75075,20.5,114.802537,78228.304829
6,76076,20.5,0.000000,NaN
7,77077,20.5,23001.885212,86585.277893
8,78078,20.5,60.003584,34019.988854
9,79079,20.5,-1725.343183,94094.177562


In [26]:
# Get the subsector total variables
subsector_total_vars = [c for c in wide_inputs_outputs_df.columns if "emission_co2e_subsector_total" in c]

In [27]:
# Filter to only subsector total columns and primary_id, time_period
la_emissions_df = wide_inputs_outputs_df[["primary_id", "time_period"] + subsector_total_vars]
la_emissions_df.head()

,primary_id,time_period,emission_co2e_subsector_total_agrc,emission_co2e_subsector_total_ccsq,emission_co2e_subsector_total_entc,emission_co2e_subsector_total_fgtv,emission_co2e_subsector_total_frst,emission_co2e_subsector_total_inen,emission_co2e_subsector_total_ippu,emission_co2e_subsector_total_lndu,emission_co2e_subsector_total_lsmm,emission_co2e_subsector_total_lvst,emission_co2e_subsector_total_scoe,emission_co2e_subsector_total_soil,emission_co2e_subsector_total_trns,emission_co2e_subsector_total_trww,emission_co2e_subsector_total_waso
0,0,6,2.924249,0.0,35.431379,13.094104,-36.545088,117.042622,4.420175,-0.077001,0.924671,0.845459,4.491483,1.233181,45.130223,0.447204,3.138402
1,0,7,2.898262,0.0,37.826585,13.180753,-36.192022,114.814166,4.438205,-0.085696,0.921864,0.832718,4.524275,1.229696,45.773554,0.453768,3.184747
2,0,8,2.839575,0.0,36.883470,13.251376,-35.881552,112.340497,4.459631,-0.094376,0.919404,0.820178,4.558470,1.222190,46.519100,0.460676,3.236670
3,0,9,2.886104,0.0,36.214098,13.395631,-35.659593,113.155278,4.484602,-0.103039,0.917305,0.807832,4.594321,1.209600,47.364325,0.467913,3.287548
4,0,10,2.874116,0.0,35.843600,13.372830,-35.172236,112.716001,4.512940,-0.111685,0.915464,0.795679,4.631982,1.191194,48.289725,0.475405,3.339940


In [28]:
# Sum all the subsector emission columns across axis=1
la_emission_total_df = la_emissions_df.copy()
la_emission_total_df["emission_total"] = la_emission_total_df[subsector_total_vars].sum(axis=1)
la_emission_total_df.head()

,primary_id,time_period,emission_co2e_subsector_total_agrc,emission_co2e_subsector_total_ccsq,emission_co2e_subsector_total_entc,emission_co2e_subsector_total_fgtv,emission_co2e_subsector_total_frst,emission_co2e_subsector_total_inen,emission_co2e_subsector_total_ippu,emission_co2e_subsector_total_lndu,emission_co2e_subsector_total_lsmm,emission_co2e_subsector_total_lvst,emission_co2e_subsector_total_scoe,emission_co2e_subsector_total_soil,emission_co2e_subsector_total_trns,emission_co2e_subsector_total_trww,emission_co2e_subsector_total_waso,emission_total
0,0,6,2.924249,0.0,35.431379,13.094104,-36.545088,117.042622,4.420175,-0.077001,0.924671,0.845459,4.491483,1.233181,45.130223,0.447204,3.138402,192.501062
1,0,7,2.898262,0.0,37.826585,13.180753,-36.192022,114.814166,4.438205,-0.085696,0.921864,0.832718,4.524275,1.229696,45.773554,0.453768,3.184747,193.800876
2,0,8,2.839575,0.0,36.883470,13.251376,-35.881552,112.340497,4.459631,-0.094376,0.919404,0.820178,4.558470,1.222190,46.519100,0.460676,3.236670,191.535310
3,0,9,2.886104,0.0,36.214098,13.395631,-35.659593,113.155278,4.484602,-0.103039,0.917305,0.807832,4.594321,1.209600,47.364325,0.467913,3.287548,193.021925
4,0,10,2.874116,0.0,35.843600,13.372830,-35.172236,112.716001,4.512940,-0.111685,0.915464,0.795679,4.631982,1.191194,48.289725,0.475405,3.339940,193.674955


In [29]:
# Keep only the primary_id, time_period, and emission_total columns
la_emission_total_df = la_emission_total_df[["primary_id", "time_period", "emission_total"]]
la_emission_total_df.head()

,primary_id,time_period,emission_total
0,0,6,192.501062
1,0,7,193.800876
2,0,8,191.535310
3,0,9,193.021925
4,0,10,193.674955


In [30]:
# aggregate data by primary_id summing the emissions
la_emission_df_sum_agg = la_emission_total_df.groupby(["primary_id"]).sum().reset_index()

# Drop time period column
la_emission_df_sum_agg = la_emission_df_sum_agg.drop(columns=["time_period"])
la_emission_df_sum_agg.head()

,primary_id,emission_total
0,0,7125.988353
1,70070,3312.632720
2,71071,4526.316959
3,72072,3107.690624
4,73073,4337.867667


In [31]:
# Filter out rows with time_period < 31
la_filtered_emission_total_df = la_emission_total_df[la_emission_total_df["time_period"] >= 31]
la_filtered_emission_total_df = la_filtered_emission_total_df.reset_index(drop=True)
la_filtered_emission_total_df.head(7)

,primary_id,time_period,emission_total
0,0,31,282.622022
1,0,32,288.677949
2,0,33,295.069525
3,0,34,301.793655
4,0,35,306.948066
5,70070,31,20.197789
6,70070,32,14.536903


In [32]:
# aggregate data by primary_id by summing the emissions
la_emission_df_mean_agg = la_filtered_emission_total_df.groupby(["primary_id"]).mean().reset_index()

# Rename emission_total to emission_avg_last_five_years
la_emission_df_mean_agg.rename(columns={"emission_total": "emission_avg_last_five_years"}, inplace=True)
la_emission_df_mean_agg

,primary_id,time_period,emission_avg_last_five_years
0,0,33.0,295.022243
1,70070,33.0,9.279604
2,71071,33.0,91.205390
3,72072,33.0,8.889596
4,73073,33.0,91.267665
5,74074,33.0,240.856073
6,75075,33.0,241.137748
7,76076,33.0,295.022243
8,77077,33.0,154.919211
9,78078,33.0,238.005883


In [33]:
# Drop year column as it is no longer needed
la_emission_df_mean_agg = la_emission_df_mean_agg.drop(columns=["time_period"])
la_emission_df_mean_agg.head()

,primary_id,emission_avg_last_five_years
0,0,295.022243
1,70070,9.279604
2,71071,91.205390
3,72072,8.889596
4,73073,91.267665


In [34]:
complete_merged_df = pd.merge(la_emission_df_mean_agg, la_emission_df_sum_agg, on=[ "primary_id"], how="inner")
complete_merged_df = pd.merge(complete_merged_df, lsu_data_agg, on=[ "primary_id"], how="inner")
complete_merged_df = pd.merge(complete_merged_df, la_production_cost_df_sum_agg, on=[ "primary_id"], how="inner")
complete_merged_df = pd.merge(complete_merged_df, la_production_df_sum_agg, on=[ "primary_id"], how="inner")

In [35]:
complete_merged_df

,primary_id,emission_avg_last_five_years,emission_total,time,num_jobs,earning_per_job,production_cost_total,production_total
0,70070,9.279604,3312.632720,20.5,24691.438413,79238.771877,6.458774e+08,398.137913
1,71071,91.205390,4526.316959,20.5,16480.194664,76819.453052,6.692580e+08,436.980824
2,72072,8.889596,3107.690624,20.5,11844.307150,95135.541477,6.458774e+08,398.137913
3,73073,91.267665,4337.867667,20.5,7673.895887,66698.88536,6.692580e+08,436.980824
4,74074,240.856073,6406.458508,20.5,2746.090115,69910.918759,6.293694e+08,454.482684
5,75075,241.137748,6240.819684,20.5,114.802537,78228.304829,6.293694e+08,454.482684
6,76076,295.022243,7125.988353,20.5,0.000000,NaN,6.293694e+08,454.482684
7,77077,154.919211,5164.408095,20.5,23001.885212,86585.277893,6.472839e+08,399.095970
8,78078,238.005883,6364.779335,20.5,60.003584,34019.988854,5.516786e+08,399.095970
9,79079,216.176892,6109.743878,20.5,-1725.343183,94094.177562,6.293694e+08,454.482684


In [52]:
cb_df = pd.read_csv(os.path.join(TORNADO_SIM_DIR, "cb_louisiana_leap_wide.csv"))

In [53]:
att_primary = pd.read_csv('C:\\Users\\pkane\\sspla\\ssp_louisiana\\jobs_data_prep\\data\\simulations\\data_for_LSU\\ATTRIBUTE_PRIMARY.csv')
att_strategy = pd.read_csv('C:\\Users\\pkane\\sspla\\ssp_louisiana\\jobs_data_prep\\data\\simulations\\data_for_LSU\\ATTRIBUTE_STRATEGY.csv')

In [54]:
cb_df = pd.merge(cb_df, att_strategy, on=[ "strategy_code"], how='inner')
cb_df = pd.merge(cb_df, att_primary[['strategy_id', 'primary_id']], on=['strategy_id'], how='inner')

In [55]:
cb_df[cb_df['primary_id']==76076]

,future_id,strategy_code,Year,air_pollution,congestion,consumer_savings,crop_value,ecosystem_services,env_pollution,fuel_cost,...,system_cost,technical_cost,technical_savings,water_pollution,strategy_id,strategy,description,transformation_specification,baseline_strategy_id,primary_id
180,0.0,PFLO:TWI_ELECTRIC_POWER_EFFICIENCY,2021.0,0.0,0.0,NaN,0.0,0.0,0.0,0.0,...,0.0,0.0,NaN,0.0,6010,TWI energy Subsector Electric Power Efficiency,Defined by TWI 20250814 - TWI energy Subsector...,TX:ENTC:DEC_LOSSES,0,76076
181,0.0,PFLO:TWI_ELECTRIC_POWER_EFFICIENCY,2022.0,0.0,0.0,NaN,0.0,0.0,0.0,0.0,...,0.0,0.0,NaN,0.0,6010,TWI energy Subsector Electric Power Efficiency,Defined by TWI 20250814 - TWI energy Subsector...,TX:ENTC:DEC_LOSSES,0,76076
182,0.0,PFLO:TWI_ELECTRIC_POWER_EFFICIENCY,2023.0,0.0,0.0,NaN,0.0,0.0,0.0,0.0,...,0.0,0.0,NaN,0.0,6010,TWI energy Subsector Electric Power Efficiency,Defined by TWI 20250814 - TWI energy Subsector...,TX:ENTC:DEC_LOSSES,0,76076
183,0.0,PFLO:TWI_ELECTRIC_POWER_EFFICIENCY,2024.0,0.0,0.0,NaN,0.0,0.0,0.0,0.0,...,0.0,0.0,NaN,0.0,6010,TWI energy Subsector Electric Power Efficiency,Defined by TWI 20250814 - TWI energy Subsector...,TX:ENTC:DEC_LOSSES,0,76076
184,0.0,PFLO:TWI_ELECTRIC_POWER_EFFICIENCY,2025.0,0.0,0.0,NaN,0.0,0.0,0.0,0.0,...,0.0,0.0,NaN,0.0,6010,TWI energy Subsector Electric Power Efficiency,Defined by TWI 20250814 - TWI energy Subsector...,TX:ENTC:DEC_LOSSES,0,76076
185,0.0,PFLO:TWI_ELECTRIC_POWER_EFFICIENCY,2026.0,0.0,0.0,NaN,0.0,0.0,0.0,0.0,...,0.0,0.0,NaN,0.0,6010,TWI energy Subsector Electric Power Efficiency,Defined by TWI 20250814 - TWI energy Subsector...,TX:ENTC:DEC_LOSSES,0,76076
186,0.0,PFLO:TWI_ELECTRIC_POWER_EFFICIENCY,2027.0,0.0,0.0,NaN,0.0,0.0,0.0,0.0,...,0.0,0.0,NaN,0.0,6010,TWI energy Subsector Electric Power Efficiency,Defined by TWI 20250814 - TWI energy Subsector...,TX:ENTC:DEC_LOSSES,0,76076
187,0.0,PFLO:TWI_ELECTRIC_POWER_EFFICIENCY,2028.0,0.0,0.0,NaN,0.0,0.0,0.0,0.0,...,0.0,0.0,NaN,0.0,6010,TWI energy Subsector Electric Power Efficiency,Defined by TWI 20250814 - TWI energy Subsector...,TX:ENTC:DEC_LOSSES,0,76076
188,0.0,PFLO:TWI_ELECTRIC_POWER_EFFICIENCY,2029.0,0.0,0.0,NaN,0.0,0.0,0.0,0.0,...,0.0,0.0,NaN,0.0,6010,TWI energy Subsector Electric Power Efficiency,Defined by TWI 20250814 - TWI energy Subsector...,TX:ENTC:DEC_LOSSES,0,76076
189,0.0,PFLO:TWI_ELECTRIC_POWER_EFFICIENCY,2030.0,0.0,0.0,NaN,0.0,0.0,0.0,0.0,...,0.0,0.0,NaN,0.0,6010,TWI energy Subsector Electric Power Efficiency,Defined by TWI 20250814 - TWI energy Subsector...,TX:ENTC:DEC_LOSSES,0,76076


In [40]:
# Make all column names lowercase
cb_df.columns = [c.lower() for c in cb_df.columns]

# Filter to only important cb columns
cb_df = cb_df[["primary_id",
               "future_id",
               "year",
               "technical_cost",
               #"consumer_savings",
               #"human_health",
               "air_pollution"]]

In [50]:
cb_df[cb_df['primary_id']==76076]

,primary_id,future_id,year,technical_cost,air_pollution
180,76076,0.0,2021.0,0.0,0.0
181,76076,0.0,2022.0,0.0,0.0
182,76076,0.0,2023.0,0.0,0.0
183,76076,0.0,2024.0,0.0,0.0
184,76076,0.0,2025.0,0.0,0.0
185,76076,0.0,2026.0,0.0,0.0
186,76076,0.0,2027.0,0.0,0.0
187,76076,0.0,2028.0,0.0,0.0
188,76076,0.0,2029.0,0.0,0.0
189,76076,0.0,2030.0,0.0,0.0


In [41]:
cb_df_agg = cb_df.groupby(["primary_id", "future_id"]).mean().reset_index()
cb_df_agg

,primary_id,future_id,year,technical_cost,air_pollution
0,70070,0.0,2035.5,-8.868730,1.536701
1,71071,0.0,2035.5,-6.086069,1.292092
2,72072,0.0,2035.5,-5.296463,1.546154
3,73073,0.0,2035.5,-3.724635,1.300412
4,74074,0.0,2035.5,-1.210177,0.032496
5,75075,0.0,2035.5,-0.626930,0.039695
6,76076,0.0,2035.5,0.000000,0.000000
7,77077,0.0,2035.5,-4.933832,0.201099
8,78078,0.0,2035.5,-0.801745,0.103703
9,79079,0.0,2035.5,-0.998483,1.294685


In [42]:
# Drop year column as it is no longer needed
cb_df_agg = cb_df_agg.drop(columns=["year"], errors='ignore')
cb_df_agg.head()

,primary_id,future_id,technical_cost,air_pollution
0,70070,0.0,-8.868730,1.536701
1,71071,0.0,-6.086069,1.292092
2,72072,0.0,-5.296463,1.546154
3,73073,0.0,-3.724635,1.300412
4,74074,0.0,-1.210177,0.032496


In [43]:
complete_merged_df = pd.merge(complete_merged_df, cb_df_agg, on=[ "primary_id"], how="inner")

In [44]:
complete_merged_df

,primary_id,emission_avg_last_five_years,emission_total,time,num_jobs,earning_per_job,production_cost_total,production_total,future_id,technical_cost,air_pollution
0,70070,9.279604,3312.632720,20.5,24691.438413,79238.771877,6.458774e+08,398.137913,0.0,-8.868730,1.536701
1,71071,91.205390,4526.316959,20.5,16480.194664,76819.453052,6.692580e+08,436.980824,0.0,-6.086069,1.292092
2,72072,8.889596,3107.690624,20.5,11844.307150,95135.541477,6.458774e+08,398.137913,0.0,-5.296463,1.546154
3,73073,91.267665,4337.867667,20.5,7673.895887,66698.88536,6.692580e+08,436.980824,0.0,-3.724635,1.300412
4,74074,240.856073,6406.458508,20.5,2746.090115,69910.918759,6.293694e+08,454.482684,0.0,-1.210177,0.032496
5,75075,241.137748,6240.819684,20.5,114.802537,78228.304829,6.293694e+08,454.482684,0.0,-0.626930,0.039695
6,76076,295.022243,7125.988353,20.5,0.000000,NaN,6.293694e+08,454.482684,0.0,0.000000,0.000000
7,77077,154.919211,5164.408095,20.5,23001.885212,86585.277893,6.472839e+08,399.095970,0.0,-4.933832,0.201099
8,78078,238.005883,6364.779335,20.5,60.003584,34019.988854,5.516786e+08,399.095970,0.0,-0.801745,0.103703
9,79079,216.176892,6109.743878,20.5,-1725.343183,94094.177562,6.293694e+08,454.482684,0.0,-0.998483,1.294685


In [45]:
complete_merged_df.to_csv('C:\\Users\\pkane\\sspla\\ssp_louisiana\\jobs_data_prep\\data\\simulations\\data_for_LSU\\sector_runs_plotting.csv', index=False)